## Growth Quantification and Mapping Barcodes

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
from dask.distributed import wait

import scipy.stats
from sklearn.linear_model import LinearRegression

from matplotlib import pyplot as plt
import holoviews as hv
hv.extension('bokeh')

In [ ]:
# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

### Functions for this notebook

In [ ]:
def filter_df(df,query_list,client=False,repartition=False,persist=False):
    #filter_list must be in df.query format (see pandas docs)
    
    #returns persisted dataframe either in cluster or local
    
    compiled_query = ' and '.join(query_list)
    out_df = df.query(compiled_query)
    if persist:
        if client:
            out_df = client.daskclient.persist(out_df)
        else:
            out_df = out_df.persist()
        
    if repartition:
        init_size = len(df)
        final_size = len(out_df)
        ratio = init_size//final_size
        out_df = out_df.repartition(npartitions=(df.npartitions // ratio) + 1)
        if persist:
            if client:
                out_df = client.daskclient.persist(out_df)
            else:
                out_df = out_df.persist()

    return out_df

def get_first_cell_timepoint(df):
    min_tpts = df.groupby(['Global CellID'])['timepoints'].idxmin().tolist()
    init_cells = df.loc[min_tpts]
    return init_cells

def get_last_cell_timepoint(df):
    max_tpts = df.groupby(['Global CellID'])['timepoints'].idxmax().tolist()
    fin_cells = df.loc[max_tpts]
    return fin_cells

def define_cell_cycle_timepoints_df(query,size_metrics):
    cell_cycle_timepoints_df = query[["timepoints","time (s)","trenchid"] + size_metrics + ["mCherry-Penta mean_intensity"]]
    cell_cycle_timepoints_df["Cell Cycle timepoints"] = cell_cycle_timepoints_df.groupby("Global CellID")["timepoints"].apply(lambda x: x - x.iloc[0])
    cell_cycle_timepoints_df = cell_cycle_timepoints_df.drop("timepoints", axis=1)
    cell_cycle_timepoints_df = cell_cycle_timepoints_df.rename({"time (s)":"Observation time (s)"}, axis=1)
    cell_cycle_timepoints_df = cell_cycle_timepoints_df.reset_index(drop=False).set_index(["Global CellID","Cell Cycle timepoints"])
    
    return cell_cycle_timepoints_df

def compile_ols_arrs(kymo_df,min_fov,max_fov):
    min_fov_idx = int(f'{int(min_fov):08n}' + '000000000000')
    max_fov_idx = int(f'{int(max_fov):08n}' + '999999999999')
    selected_kymo_df = kymo_df.loc[min_fov_idx:max_fov_idx]
    timepoint_df = selected_kymo_df.groupby(["fov","timepoints"])["time (s)"].apply(lambda x: x.iloc[0]).compute(scheduler='threads')
    timepoint_df = pd.DataFrame(timepoint_df).sort_index()
    timepoint_df["time (s)^2"] = timepoint_df["time (s)"]**2
    fov_list = timepoint_df.index.get_level_values(0).unique().tolist()
    time_arr = np.array([timepoint_df["time (s)"].loc[f].values for f in fov_list])
    time_arr_accum = np.add.accumulate(time_arr,axis=1)
    time_sq_arr = np.array([timepoint_df["time (s)^2"].loc[f].values for f in fov_list])
    time_sq_arr_accum = np.add.accumulate(time_sq_arr,axis=1)
    return fov_list,time_arr,time_arr_accum,time_sq_arr_accum    
    

def fit_least_squares(log_size_arr,fov_idx_in_arr,time_i,time_f,time_arr,time_arr_accum,time_sq_arr_accum):
    # everything assumes an inclusive time_f
    if time_i>0:
        t_sum = time_arr_accum[fov_idx_in_arr,time_f]-time_arr_accum[fov_idx_in_arr,time_i-1]
        t_sq_sum = time_sq_arr_accum[fov_idx_in_arr,time_f]-time_sq_arr_accum[fov_idx_in_arr,time_i-1]
        
    else:
        t_sum = time_arr_accum[fov_idx_in_arr,time_f]
        t_sq_sum = time_sq_arr_accum[fov_idx_in_arr,time_f]
    
    n_obs = time_f-time_i+1
    
    size_sum = np.sum(log_size_arr)
    time_size_sum = np.sum(log_size_arr*time_arr[fov_idx_in_arr,time_i:time_f+1])
    
    denom = (n_obs*t_sq_sum) - (t_sum**2)
    num = (n_obs*time_size_sum) - (t_sum*size_sum)
    
    beta = num/denom
    alpha = np.mean(log_size_arr)-beta*(t_sum/n_obs)
    return beta,alpha


def fit_least_squares_windowed(timepoint_arr,log_size_arr,fov_idx_in_arr,time_arr,time_arr_accum,time_sq_arr_accum,ols_window_size=3):
    timepoint_arr_len = len(timepoint_arr)
    timepoint_window_arr = [timepoint_arr[i:i+ols_window_size] for i in range(0,timepoint_arr_len-ols_window_size+1,1)]
    timepoint_window_arr = np.array([[item[0],item[-1]] for item in timepoint_window_arr])
    beta_output = []
    time_output = []
    for timepoint_idx,timepoint_window in enumerate(timepoint_window_arr):
        time_i = timepoint_window[0]
        time_f = timepoint_window[1]
        windowed_log_size_arr = log_size_arr[timepoint_idx:timepoint_idx+ols_window_size]
        beta,_ = fit_least_squares(windowed_log_size_arr,fov_idx_in_arr,time_i,time_f,time_arr,time_arr_accum,time_sq_arr_accum)
        beta_output.append(beta)
        time_output.append(np.mean(time_arr[fov_idx_in_arr,time_i:time_f+1]))
    return beta_output,time_output

def fit_least_squares_windowed_cellgroupby(cell_group,growth_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum,ols_window_size=3):
    timepoint_arr = cell_group["timepoints"].values
    log_size_arr = np.log(cell_group[growth_metric].values)
    fov_idx = cell_group["fov"].iloc[0]
    fov_idx_in_arr = np.where(np.array(fov_list)==fov_idx)[0][0]
    
    beta_output,time_output = fit_least_squares_windowed(timepoint_arr,log_size_arr,fov_idx_in_arr,time_arr,time_arr_accum,time_sq_arr_accum,ols_window_size=ols_window_size)
    growth_rate_output = np.array([beta_output,time_output]).T
    growth_rate_output = growth_rate_output.tolist()
    return growth_rate_output

def fit_least_squares_cellcycle(timepoint_arr,log_size_arr,fov_idx_in_arr,time_arr,time_arr_accum,time_sq_arr_accum):
    time_i = timepoint_arr[0]
    time_f = timepoint_arr[-1]
    beta_output,_ = fit_least_squares(log_size_arr,fov_idx_in_arr,time_i,time_f,time_arr,time_arr_accum,time_sq_arr_accum)
    time_output = np.mean(time_arr[fov_idx_in_arr,time_i:time_f+1])
    
    return beta_output,time_output

def fit_least_squares_cellcycle_cellgroupby(cell_group,size_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum):
    timepoint_arr = cell_group["timepoints"].values
    if len(timepoint_arr)>1:
        log_size_arr = np.log(cell_group[size_metric].values)
        fov_idx = cell_group["fov"].iloc[0]
        fov_idx_in_arr = np.where(np.array(fov_list)==fov_idx)[0][0]
        beta_output,time_output = fit_least_squares_cellcycle(timepoint_arr,log_size_arr,fov_idx_in_arr,time_arr,time_arr_accum,time_sq_arr_accum)

        growth_rate_output = [beta_output,time_output]
    else:
        growth_rate_output = []
    return growth_rate_output

def get_cellcycle_growth_rate_df(query,size_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum,get_measurement_time=False):
    all_gr = query.groupby(level=0,sort=False).apply(lambda x: fit_least_squares_cellcycle_cellgroupby(x,size_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum))
    trenchid_series = query.groupby(level=0,sort=False).apply(lambda x: x["trenchid"].iloc[0])
    
    if get_measurement_time:
        all_gr = pd.concat([all_gr,trenchid_series],axis=1).rename({0:"Growth Measurment",1:"trenchid"},axis=1)
    else:
        all_gr = pd.DataFrame(all_gr, columns=["Growth Measurment"])
        
    all_gr = all_gr.dropna(subset=["Growth Measurment"])
    all_gr = all_gr[all_gr["Growth Measurment"].apply(len)==2]
    
    all_gr["Growth Rate: " + size_metric] = all_gr.apply(lambda x: x["Growth Measurment"][0], axis=1)*3600 #size unit per hr
    if get_measurement_time:
        all_gr["Measurement time (s)"] = all_gr.apply(lambda x: x["Growth Measurment"][1], axis=1)
    all_gr = all_gr.drop("Growth Measurment", axis=1)
    return all_gr

def get_instantaneous_growth_rate_df(query,growth_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum,get_measurement_time=False):
    all_gr = query.groupby(level=0,sort=False).apply(lambda x: fit_least_squares_windowed_cellgroupby(x,growth_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum))
    trenchid_series = query.groupby(level=0,sort=False).apply(lambda x: x["trenchid"].iloc[0])
    
    if get_measurement_time:
        all_gr = pd.concat([all_gr,trenchid_series],axis=1).rename({0:"Growth Measurment",1:"trenchid"},axis=1)
    else:
        all_gr = pd.DataFrame(all_gr, columns=["Growth Measurment"])
    all_gr = all_gr.explode("Growth Measurment").dropna(subset=["Growth Measurment"])
    
    all_gr["Growth Rate Measurement Index"] = all_gr.groupby("Global CellID").cumcount()
    all_gr["Instantaneous Growth Rate: " + growth_metric] = all_gr.apply(lambda x: x["Growth Measurment"][0], axis=1)*3600 #size unit per hr
    if get_measurement_time:
        all_gr["Measurement time (s)"] = all_gr.apply(lambda x: x["Growth Measurment"][1], axis=1)
    all_gr = all_gr.drop("Growth Measurment", axis=1)
    return all_gr

def get_growth_and_division_cell_cycle(lineage_df,kymo_df_path,trench_score_thr=-75,absolute_time=True,delta_t_min=None,\
                                  size_metrics=["area","major_axis_length","minor_axis_length","Volume","Surface Area"]):
    
    ## Setting up kymograph and lineage dfs
    
    kymo_df = dd.read_parquet(kymo_df_path,calculate_divisions=True)
    kymo_idx_list = lineage_df["Kymograph FOV Parquet Index"].tolist()
    
    if not absolute_time:
        kymo_df["time (s)"] = kymo_df["timepoints"]*delta_t_min*60.
    
    kymo_time_series = kymo_df["time (s)"].loc[kymo_idx_list].compute(scheduler='threads')
    kymo_time_series.index = lineage_df.index
    lineage_df["time (s)"] = kymo_time_series
            
    reference = filter_df(lineage_df,["`Trench Score` < " + str(trench_score_thr)])
    query = filter_df(lineage_df,["`Mother CellID` != -1","`Daughter CellID 1` != -1","`Daughter CellID 2` != -1",\
                                                  "`Sister CellID` != -1","`Trench Score` < " + str(trench_score_thr)])
    
    init_cells = get_first_cell_timepoint(query).reset_index().set_index("Global CellID").sort_index()
    fin_cells = get_last_cell_timepoint(query).reset_index().set_index("Global CellID").sort_index()

    cell_min_tpt_df = get_first_cell_timepoint(reference).reset_index().set_index("Global CellID").sort_index()
    cell_max_tpt_df = get_last_cell_timepoint(reference).reset_index().set_index("Global CellID").sort_index()

    mother_df = cell_max_tpt_df.loc[init_cells["Mother CellID"].tolist()]
    sister_df = cell_min_tpt_df.loc[init_cells["Sister CellID"].tolist()]
    daughter_1_df = cell_min_tpt_df.loc[fin_cells["Daughter CellID 1"].tolist()]
    daughter_2_df = cell_min_tpt_df.loc[fin_cells["Daughter CellID 2"].tolist()]

    for metric in size_metrics:

        if metric == "minor_axis_length":

            init_cells["Birth: " + metric] = init_cells[metric].values
            init_cells["Division: " + metric] = fin_cells[metric].values
            init_cells["Delta: " + metric] = fin_cells[metric].values-init_cells[metric].values

        else:

            interp_mother_final_size = ((init_cells[metric].values + sister_df[metric].values) * mother_df[metric].values)**(1/2)
            sister_frac = init_cells[metric].values/(sister_df[metric].values+init_cells[metric].values)
            init_cells["Birth: " + metric] = (sister_frac*interp_mother_final_size)
            init_cells["Division: " + metric] = (((daughter_1_df[metric].values + daughter_2_df[metric].values) * fin_cells[metric].values)**(1/2))
            init_cells["Delta: " + metric] = init_cells["Division: " + metric].values-init_cells["Birth: " + metric].values
            
            if metric == "major_axis_length":
                init_cells["Septum Displacement"] = np.abs(daughter_1_df[metric].values-daughter_2_df[metric].values)
                init_cells["Septum Displacement Length Normalized"] = np.abs(daughter_1_df[metric].values-daughter_2_df[metric].values)/(daughter_1_df[metric].values+daughter_2_df[metric].values)

    init_cells["Final timepoints"] = daughter_1_df["timepoints"].values #counting a timepoint in which a division occurs as a full timepoint, hacky
    init_cells["Delta Timepoints"] = init_cells["Final timepoints"]-init_cells["timepoints"]

    # if absolute_time:
    interpolated_final_time = (fin_cells["time (s)"].values+daughter_1_df["time (s)"].values)/2 #interpolating under the same assumptions as the size quantification
    interpolated_init_time = ((init_cells["time (s)"].values + mother_df["time (s)"].values)/2)
    init_cells["Final time (s)"] = interpolated_final_time
    init_cells["Initial time (s)"] = interpolated_init_time
    init_cells["Delta time (s)"] = interpolated_final_time-interpolated_init_time
    init_cells["Mid-Cycle time (s)"] = (interpolated_init_time+interpolated_final_time)/2

    query = query.reset_index().set_index(["Global CellID","timepoints"]).sort_index().reset_index(level=1)
    
    delta_t_series = query.groupby(level=0,sort=False)["time (s)"].apply(lambda x: ((x[1:].values - x[:-1].values)))

    init_time_gap = init_cells["time (s)"].values - interpolated_init_time
    final_time_gap = interpolated_final_time - fin_cells["time (s)"].values
    
    init_cells = init_cells.rename(columns={"timepoints":"initial timepoints"})
    
    ### Growth rate quantification
    fov_list = query["fov"].unique().tolist()
    min_fov = fov_list[0]
    max_fov = fov_list[-1]

    fov_list,time_arr,time_arr_accum,time_sq_arr_accum = compile_ols_arrs(kymo_df,min_fov,max_fov)
    
    gr_df_list = []
    for size_idx,size_metric in enumerate(size_metrics): # Havn't decided between mean and median
        if size_idx == 0:
            gr_df = get_cellcycle_growth_rate_df(query,size_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum,get_measurement_time=True)
        else:
            gr_df = get_cellcycle_growth_rate_df(query,size_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum,get_measurement_time=False)

        gr_df_list.append(gr_df)

    gr_df_out = pd.concat(gr_df_list,axis=1)
    gr_df_out = gr_df_out.drop(["trenchid","Measurement time (s)"],axis=1)
    
    init_cells = init_cells.join(gr_df_out)
    
    return init_cells

def get_growth_and_division_cell_cycle_timepoints(lineage_df,kymo_df_path,trench_score_thr=-75,absolute_time=True,delta_t_min=None,\
                                  size_metrics=["area","major_axis_length","minor_axis_length","Volume","Surface Area"]):
    
    ## Setting up kymograph and lineage dfs
    
    kymo_df = dd.read_parquet(kymo_df_path,calculate_divisions=True)
    kymo_idx_list = lineage_df["Kymograph FOV Parquet Index"].tolist()
    
    if not absolute_time:
        kymo_df["time (s)"] = kymo_df["timepoints"]*delta_t_min*60.
    
    kymo_time_series = kymo_df["time (s)"].loc[kymo_idx_list].compute(scheduler='threads')
    kymo_time_series.index = lineage_df.index
    lineage_df["time (s)"] = kymo_time_series
            
    query = filter_df(lineage_df,["`Mother CellID` != -1","`Daughter CellID 1` != -1","`Daughter CellID 2` != -1",\
                                                  "`Sister CellID` != -1","`Trench Score` < " + str(trench_score_thr)])

    query = query.reset_index().set_index(["Global CellID","timepoints"]).sort_index().reset_index(level=1)
    
    cell_cycle_timepoints_df = define_cell_cycle_timepoints_df(query,size_metrics)
    
    cellid_cell_cycle_timepoint_idx = cell_cycle_timepoints_df.reset_index().apply(lambda x: int(f'{int(x["Global CellID"]):016n}{int(x["Cell Cycle timepoints"]):04n}'), axis=1).tolist()
    cell_cycle_timepoints_df["Global CellID-Cell Cycle timepoints"] = cellid_cell_cycle_timepoint_idx
    cell_cycle_timepoints_df = cell_cycle_timepoints_df.reset_index().set_index("Global CellID-Cell Cycle timepoints")
    
    return cell_cycle_timepoints_df

def get_growth_and_division_growth_rate(lineage_df,kymo_df_path,trench_score_thr=-75,absolute_time=True,delta_t_min=None,\
                                  growth_metrics=["area","major_axis_length","Volume","Surface Area"]):

    kymo_df = dd.read_parquet(kymo_df_path,calculate_divisions=True)
    kymo_idx_list = lineage_df["Kymograph FOV Parquet Index"].tolist()

    if not absolute_time:
        kymo_df["time (s)"] = kymo_df["timepoints"]*delta_t_min*60.

    kymo_time_series = kymo_df["time (s)"].loc[kymo_idx_list].compute(scheduler='threads')
    kymo_time_series.index = lineage_df.index
    lineage_df["time (s)"] = kymo_time_series

    query = filter_df(lineage_df,["`Mother CellID` != -1","`Daughter CellID 1` != -1","`Daughter CellID 2` != -1",\
                                                  "`Sister CellID` != -1","`Trench Score` < " + str(trench_score_thr)])

    query = query.reset_index().set_index(["Global CellID","timepoints"]).sort_index().reset_index(level=1)
    
    fov_list = query["fov"].unique().tolist()
    min_fov = fov_list[0]
    max_fov = fov_list[-1]

    fov_list,time_arr,time_arr_accum,time_sq_arr_accum = compile_ols_arrs(kymo_df,min_fov,max_fov)

    gr_df_list = []
    for growth_idx,growth_metric in enumerate(growth_metrics): # Havn't decided between mean and median
        if growth_idx == 0:
            gr_df = get_instantaneous_growth_rate_df(query,growth_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum,get_measurement_time=True)
        else:
            gr_df = get_instantaneous_growth_rate_df(query,growth_metric,fov_list,time_arr,time_arr_accum,time_sq_arr_accum,get_measurement_time=False)

        gr_df["Global CellID-Growth Rate Measurement Index"] = gr_df.reset_index().apply(lambda x: int(f'{int(x["Global CellID"]):016n}{int(x["Growth Rate Measurement Index"]):04n}'), axis=1).tolist()
        gr_df = gr_df.reset_index()
        if growth_idx != 0:
            gr_df = gr_df.drop(["Global CellID","Growth Rate Measurement Index"],axis=1)
        gr_df = gr_df.set_index("Global CellID-Growth Rate Measurement Index")
        gr_df_list.append(gr_df)

        # Initial/Final Growth Rate numbers are probably untrustworthy since noise in delta t from the division 
        # event is probably pretty high. Going to exclude.
    gr_df_out = pd.concat(gr_df_list,axis=1)
    
    return gr_df_out

def get_all_growth_and_division_stats(lineage_df,kymo_df_path,trench_score_thr=-75,absolute_time=True,delta_t_min=None,\
                                      size_metrics=["area","major_axis_length","minor_axis_length","Volume","Surface Area"],\
                                     growth_metrics=["area","major_axis_length","Volume","Surface Area"]):
    input_test_partition = lineage_df.get_partition(0).compute()
    
    test_partition_1 = get_growth_and_division_cell_cycle(input_test_partition,kymo_df_path,\
                                    trench_score_thr=trench_score_thr,absolute_time=absolute_time,\
                                    delta_t_min=delta_t_min,size_metrics=size_metrics)
    growth_div_df = dd.map_partitions(get_growth_and_division_cell_cycle,lineage_df,kymo_df_path,\
                                      trench_score_thr=trench_score_thr,absolute_time=absolute_time,delta_t_min=delta_t_min,\
                                      size_metrics=size_metrics,meta=test_partition_1)
    
    test_partition_2 = get_growth_and_division_cell_cycle_timepoints(input_test_partition,kymo_df_path,\
                                    trench_score_thr=trench_score_thr,absolute_time=absolute_time,\
                                    delta_t_min=delta_t_min,size_metrics=size_metrics)
    cell_cycle_timepoints_df = dd.map_partitions(get_growth_and_division_cell_cycle_timepoints,lineage_df,kymo_df_path,\
                                      trench_score_thr=trench_score_thr,absolute_time=absolute_time,delta_t_min=delta_t_min,\
                                      size_metrics=size_metrics,meta=test_partition_2)
    
    test_partition_3 = get_growth_and_division_growth_rate(input_test_partition,kymo_df_path,\
                                    trench_score_thr=trench_score_thr,absolute_time=absolute_time,\
                                    delta_t_min=delta_t_min,growth_metrics=growth_metrics)
    cell_cycle_growth_df = dd.map_partitions(get_growth_and_division_growth_rate,lineage_df,kymo_df_path,\
                                      trench_score_thr=trench_score_thr,absolute_time=absolute_time,delta_t_min=delta_t_min,\
                                      growth_metrics=growth_metrics,meta=test_partition_3)
        
    return growth_div_df,cell_cycle_timepoints_df,cell_cycle_growth_df

def get_barcode_pheno_kymo_df(phenotype_df, phenotype_kymo_df, barcode_df, trenchid_map, output_kymo_index="FOV Parquet Index"):
    ## phenotype_df_list must contain dfs with trenchids column
    ## can be made more effecient still, with a direct output to parquet
    ## More effecient implementation that splits outputs into two smaller parts (every cell, every trench)
    ## Saves by elmininating some redundent entries
    ## Still filters only for cells that made it to final "phenotype_df" in most cases lineage traced
    
    valid_barcode_df = barcode_df[barcode_df["trenchid"].isin(trenchid_map.keys())].compute()
    barcode_df_mapped_trenchids = valid_barcode_df["trenchid"].apply(lambda x: trenchid_map[x])
    phenotype_df_idx = phenotype_df["trenchid"].unique().compute().tolist()

    valid_init_df_indices = barcode_df_mapped_trenchids.isin(phenotype_df_idx)
    barcode_df_mapped_trenchids = barcode_df_mapped_trenchids[valid_init_df_indices]
    barcode_df_mapped_trenchids_list = barcode_df_mapped_trenchids.tolist()
    final_valid_barcode_df_indices = barcode_df_mapped_trenchids.index.to_list()

    called_df = barcode_df.loc[final_valid_barcode_df_indices]
    called_df["phenotype trenchid"] = barcode_df_mapped_trenchids
    called_df["phenotype trenchid"] = called_df["phenotype trenchid"].astype(int)
    called_df = called_df.drop(["Barcode Signal"], axis=1)
    called_df = called_df.reset_index().set_index("phenotype trenchid",drop=True,sorted=False)

    output_phenotype_kmyo_df = phenotype_kymo_df.rename(columns={"trenchid":"phenotype trenchid"})
    output_phenotype_kmyo_df = output_phenotype_kmyo_df.reset_index().set_index("phenotype trenchid",drop=True,sorted=True)
    output_phenotype_kmyo_df = output_phenotype_kmyo_df.loc[barcode_df_mapped_trenchids_list]
    called_df = called_df.repartition(divisions = output_phenotype_kmyo_df.divisions).persist()
    wait(called_df)
    output_phenotype_kmyo_df = output_phenotype_kmyo_df.merge(called_df,how='inner',left_index=True,right_index=True)
    output_phenotype_kmyo_df = output_phenotype_kmyo_df.reset_index().set_index(output_kymo_index)

    return output_phenotype_kmyo_df,barcode_df_mapped_trenchids

def get_output_phenotype_df(phenotype_df,barcode_df_mapped_trenchids,output_index="File Parquet Index"):
    
    phenotype_df_idx = phenotype_df["trenchid"].unique().compute().tolist()
    valid_init_df_indices = barcode_df_mapped_trenchids.isin(phenotype_df_idx)
    barcode_df_mapped_trenchids = barcode_df_mapped_trenchids[valid_init_df_indices]
    barcode_df_mapped_trenchids_list = barcode_df_mapped_trenchids.tolist()
    
    output_phenotype_df = phenotype_df.rename(columns={"trenchid":"phenotype trenchid"})
    output_phenotype_df = output_phenotype_df.reset_index().set_index("phenotype trenchid",drop=True,sorted=True)
    
    output_phenotype_df = output_phenotype_df.loc[barcode_df_mapped_trenchids_list]
    output_phenotype_df = output_phenotype_df.reset_index().set_index(output_index) 
    
    return output_phenotype_df

### Start Dask

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=200,
    n_workers_min=100,
    memory="4GB",#"8GB",
    working_directory="/home/lag36/scratch/lag36/",
)
dask_controller.startdask()

In [ ]:
dask_controller.displaydashboard()

### Import Lineage

In [ ]:
corrected_lineage_df = dd.read_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/2022-09-07_FOV_and_Time_Normalization/",calculate_divisions=True)

### Hack for corrupted timepoints

Here, I am overwriting the corrupted timestamps of this experiment with a duplicate experiment that should have roughly the same timepoints.

In [ ]:
from dask.distributed import wait

kymo_df = dd.read_parquet("/home/de64/scratch/de64/sync_folder/2022-01-20_lDE20_Final_6/Growth_Division/kymograph/metadata",calculate_divisions=True)
global_meta_replace = tr.pandas_hdf5_handler("/home/de64/scratch/de64/sync_folder/2021-11-08_lDE20_Final_3/Growth_Division/metadata.hdf5").read_df("global",read_metadata=False)

kymo_df["fov-timepoint_idx"] = kymo_df.apply(lambda x: int(f'{int(x["fov"]):04n}{int(x["timepoints"]):04n}'), axis=1)
kymo_df = kymo_df.reset_index().set_index("fov-timepoint_idx")
global_meta_replace = global_meta_replace.reset_index()
global_meta_replace["fov-timepoint_idx"] = global_meta_replace.apply(lambda x: int(f'{int(x["fov"]):04n}{int(x["timepoints"]):04n}'), axis=1)
global_meta_replace = global_meta_replace[["fov-timepoint_idx","t"]]
global_meta_replace = global_meta_replace.set_index("fov-timepoint_idx")
kymo_df = kymo_df.join(global_meta_replace).set_index("FOV Parquet Index")
kymo_df["time (s)"] = kymo_df["t"]
kymo_df = kymo_df.drop("t", axis=1)

dd.to_parquet(kymo_df, "/home/de64/scratch/de64/sync_folder/2022-01-20_lDE20_Final_6/Growth_Division/kymograph/metadata_timefix", engine="pyarrow", overwrite=True)

### Perform Growth/Division Quantification

In [ ]:
growth_div_df,cell_cycle_timepoints_df,cell_cycle_growth_df = get_all_growth_and_division_stats(corrected_lineage_df,"/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/metadata",\
                                                 absolute_time=True)

### Import Barcode Dataframe

- The decode rate is suprisingly decent ~70% given the incomplete codebook
- The retention rate is super low though <50%, something is going on with unloading...not sure what the fix is here...maybe a faster flow rate or more gradual entry (can experiment with that, on the next run)
- thought the phenotyping recorded 364111 trenches so its actually more like 364111/333269, so actually unloading was not a major factor, it was more a lack in initial loading

In [ ]:
meta_handle = tr.pandas_hdf5_handler("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Barcodes/barcode_output_df.hdf5")
pandas_barcode_df = meta_handle.read_df("barcodes",read_metadata=True)
barcode_df = dd.from_pandas(pandas_barcode_df,npartitions=500,sort=True)
barcode_df = barcode_df.persist()

ttl_called = len(barcode_df.index)
ttl_trenches = pandas_barcode_df.metadata['Total Trenches']
ttl_trenches_w_cells = pandas_barcode_df.metadata['Total Trenches With Cells']
percent_called = ttl_called/ttl_trenches
percent_called_w_cells = ttl_called/ttl_trenches_w_cells

print(ttl_called)
print(ttl_trenches)
print(ttl_trenches_w_cells)
print(percent_called)
print(percent_called_w_cells)

In [ ]:
def get_output_phenotype_df_nobarcodes(phenotype_df,output_index="File Parquet Index"):
    
    barcode_df_col_names = ['Barcode','consensus',
   'reference', 'alignmentstart', 'cigar', 'subsample', 'aligned_cons',
   'split_ref', 'split_align', 'hamm_ref', 'UMI', 'sgRNA',
   'Closest Hamming Distance', 'oDEPool7_id', 'EcoWG1_id', 'Gene',
   'Strand', 'Target Sequence', 'Target Sites', 'N Target Sites',
   'N Mismatch', 'Category', 'TargetID', 'barcode', 'barcodeid']
    
    output_phenotype_df = phenotype_df.rename(columns={"trenchid":"phenotype trenchid"})
    output_phenotype_df = output_phenotype_df.reset_index().set_index("phenotype trenchid",drop=True,sorted=True)
    
    for column_name in barcode_df_col_names:
        output_phenotype_df[column_name] = 0
            
    output_phenotype_df = output_phenotype_df.reset_index().set_index(output_index) 
    
    return output_phenotype_df

#### Get Trench Mapping

 - Note that if the trenches are slightly unaligned from the experiment to the decoding timeseries, you can manually insert an offset in the files_to_trenshid_map function that you determine from the images.

In [ ]:
# phenotype_kymopath = "/home/de64/scratch/de64/sync_folder/2022-01-20_lDE20_Final_6/Growth_Division/kymograph/metadata"
# barcode_kymopath = "/home/de64/scratch/de64/sync_folder/2022-01-20_lDE20_Final_6/Barcodes/kymograph/metadata"

phenotype_kymopath = "/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/metadata"
barcode_kymopath = "/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Barcodes/kymograph/metadata"

trenchid_map = tr.files_to_trenchid_map(phenotype_kymopath,barcode_kymopath, offset_x=-99*0.105951961895293, offset_y=0)
# trenchid_map = tr.files_to_trenchid_map(phenotype_kymopath,barcode_kymopath, offset_x=-99*0.11, offset_y=20*0.11)

#### Get Output Dataframe

In [ ]:
kymo_df = dd.read_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/metadata",calculate_divisions=True)

growth_div_df = growth_div_df.persist()
wait(growth_div_df)
output_phenotype_kmyo_df,barcode_df_mapped_trenchids = get_barcode_pheno_kymo_df(growth_div_df, kymo_df, barcode_df, trenchid_map)
# output_phenotype_kmyo_df.to_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_FInal_Barcodes_df_Run_1/", engine="pyarrow", overwrite=True)
# dask_controller.daskclient.cancel(output_phenotype_kmyo_df)

growth_div_df_output = get_output_phenotype_df(growth_div_df,barcode_df_mapped_trenchids,output_index="Global CellID")
# growth_div_df_output.to_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_Lineage_Cell_Cycle_Run_1/", engine="pyarrow", overwrite=True)
# dask_controller.daskclient.cancel(growth_div_df_output)
# dask_controller.daskclient.cancel(growth_div_df)

# cell_cycle_timepoints_df = cell_cycle_timepoints_df.persist()
# wait(cell_cycle_timepoints_df)
# cell_cycle_timepoints_df_output = get_output_phenotype_df(cell_cycle_timepoints_df,barcode_df_mapped_trenchids,output_index="Global CellID-Cell Cycle timepoints")
# cell_cycle_timepoints_df_output.to_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_Lineage_Observations_Run_1/", engine="pyarrow", overwrite=True)
# # dask_controller.daskclient.cancel(cell_cycle_timepoints_df_output)
# # dask_controller.daskclient.cancel(cell_cycle_timepoints_df)

# cell_cycle_growth_df = cell_cycle_growth_df.persist()
# wait(cell_cycle_growth_df)
# cell_cycle_growth_df_output = get_output_phenotype_df(cell_cycle_growth_df,barcode_df_mapped_trenchids,output_index="Global CellID-Growth Rate Measurement Index")
# cell_cycle_growth_df_output.to_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_Lineage_Growth_Observations_Run_1", engine="pyarrow", overwrite=True)
# # dask_controller.daskclient.cancel(cell_cycle_growth_df_output)
# # dask_controller.daskclient.cancel(cell_cycle_growth_df)

In [ ]:
kymo_df = dd.read_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/metadata",calculate_divisions=True)

growth_div_df = growth_div_df.persist()
wait(growth_div_df)
output_phenotype_kmyo_df,barcode_df_mapped_trenchids = get_barcode_pheno_kymo_df(growth_div_df, kymo_df, barcode_df, trenchid_map)
output_phenotype_kmyo_df.to_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_FInal_Barcodes_df_Run_1/", engine="pyarrow", overwrite=True)
# dask_controller.daskclient.cancel(output_phenotype_kmyo_df)

growth_div_df_output = get_output_phenotype_df(growth_div_df,barcode_df_mapped_trenchids,output_index="Global CellID")
growth_div_df_output.to_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_Lineage_Cell_Cycle_Run_1/", engine="pyarrow", overwrite=True)
# dask_controller.daskclient.cancel(growth_div_df_output)
# dask_controller.daskclient.cancel(growth_div_df)

cell_cycle_timepoints_df = cell_cycle_timepoints_df.persist()
wait(cell_cycle_timepoints_df)
cell_cycle_timepoints_df_output = get_output_phenotype_df(cell_cycle_timepoints_df,barcode_df_mapped_trenchids,output_index="Global CellID-Cell Cycle timepoints")
cell_cycle_timepoints_df_output.to_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_Lineage_Observations_Run_1/", engine="pyarrow", overwrite=True)
# dask_controller.daskclient.cancel(cell_cycle_timepoints_df_output)
# dask_controller.daskclient.cancel(cell_cycle_timepoints_df)

cell_cycle_growth_df = cell_cycle_growth_df.persist()
wait(cell_cycle_growth_df)
cell_cycle_growth_df_output = get_output_phenotype_df(cell_cycle_growth_df,barcode_df_mapped_trenchids,output_index="Global CellID-Growth Rate Measurement Index")
cell_cycle_growth_df_output.to_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_Lineage_Growth_Observations_Run_1", engine="pyarrow", overwrite=True)
# dask_controller.daskclient.cancel(cell_cycle_growth_df_output)
# dask_controller.daskclient.cancel(cell_cycle_growth_df)

### Quick Phenotype Check (for map)

In [ ]:
obs = growth_div_df_output
bar = output_phenotype_kmyo_df

In [ ]:
obs = dd.read_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_Lineage_Cell_Cycle_Run_1/", engine="pyarrow",calculate_divisions=True)
bar = dd.read_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_FInal_Barcodes_df_Run_1/", engine="pyarrow",calculate_divisions=True)

## -99 After Hamming distance 2

In [ ]:
bar[bar["Gene"]=="nagE"].compute()

In [ ]:
ftsn_df = bar[bar["Gene"]=="nagE"].compute()
selection_df = ftsn_df[(ftsn_df["N Mismatch"]==3.)&(ftsn_df["TargetID"]==1031)]
filtered_df = obs[obs["phenotype trenchid"].isin(selection_df["phenotype trenchid"].unique())].compute()
from matplotlib import pyplot as plt

plt.scatter(filtered_df['time (s)'],filtered_df['Division: major_axis_length'],s=5)

In [ ]:
ftsn_df = bar[bar["Gene"]=="ftsN"].compute()
selection_df = ftsn_df[(ftsn_df["N Mismatch"]==5.)&(ftsn_df["TargetID"]==875)]
filtered_df = obs[obs["phenotype trenchid"].isin(selection_df["phenotype trenchid"].unique())].compute()
from matplotlib import pyplot as plt

plt.scatter(filtered_df['time (s)'],filtered_df['Division: major_axis_length'],s=5)

In [ ]:
ftsn_df = bar[bar["Gene"]=="ftsK"].compute()
selection_df = ftsn_df[(ftsn_df["N Mismatch"]==5.)&(ftsn_df["TargetID"]==1848)]
filtered_df = obs[obs["phenotype trenchid"].isin(selection_df["phenotype trenchid"].unique())].compute()
from matplotlib import pyplot as plt

plt.scatter(filtered_df['time (s)'],filtered_df['Division: major_axis_length'],s=5)

In [ ]:
filtered_df.columns

In [ ]:
dask_controller.shutdown()

In [ ]:
ftsn_df = bar[bar["Gene"]=="ftsA"].compute()
selection_df = ftsn_df[(ftsn_df["N Mismatch"]==10.)&(ftsn_df["TargetID"]==2505)]
filtered_df = obs[obs["phenotype trenchid"].isin(selection_df["phenotype trenchid"].unique())].compute()
from matplotlib import pyplot as plt

plt.scatter(filtered_df['time (s)'],filtered_df['Division: major_axis_length'],s=5)

In [ ]:
ftsn_df = bar[bar["Gene"]=="nrdA"].compute()
selection_df = ftsn_df
# selection_df = ftsn_df[(ftsn_df["N Mismatch"]==5.)&(ftsn_df["TargetID"]==1848)]
filtered_df = obs[obs["phenotype trenchid"].isin(selection_df["phenotype trenchid"].unique())].compute()
from matplotlib import pyplot as plt

plt.scatter(filtered_df['time (s)'],filtered_df['Division: major_axis_length'],s=5)

In [ ]:
dask_controller.shutdown()